# 02 · Calibration and training

1. Fit the per-device **lens-position → distance** calibration on the training captures.
2. Train `UNetMulti` (mask + depth) with the calibrated focus distance as a side input.
3. Train the **image-only** ablation (meta input zeroed) for comparison in `03_evaluate`.

Only sessions whose manifest says `split: train` are used, and validation is a whole held-out
session. The test zip is never opened here. The model never receives depth as an input.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
EPOCHS = 60
BATCH_SIZE = 8
DEPTH_WEIGHT = 0.5          # weight of the depth loss against the mask loss
OBJECT_DEPTH_WEIGHT = 1.0   # weight of the depth error averaged over object pixels alone (0 = plain L1 over all valid pixels)
VAL_SESSIONS = None         # e.g. ["../data/session_....zip"]; None holds out about a quarter of the sessions

In [ ]:
from unetultra.session import find_sessions
from unetultra.train import split_sessions

train_paths, val_paths = split_sessions(find_sessions(DATA, "train"), VAL_SESSIONS)
print("train:", *train_paths, sep="\n  ")
print("validation:", *val_paths, sep="\n  ")

## Lens calibration

Each point is one capture: the lens reading after focus settled, against the LiDAR depth at the
tap (median of valid pixels in a small window). The fit is monotonic (isotonic regression on
1/distance). Residuals should scatter around zero with no trend.

In [ ]:
from unetultra import plots
from unetultra.calibration import fit_from_sessions

calibration, points = fit_from_sessions(train_paths)
calibration.save(MODELS / "calibration.json")
print(f"{calibration.n_samples} samples, RMSE {calibration.rmse_m * 100:.2f} cm, device {calibration.device_model}")
plots.calibration_fit(calibration, points);

## Datasets

Accepted samples only. Pre-processing takes the centre square of each frame and resizes it to
512×512, with the intrinsics transformed to match. Samples whose object was tapped outside that
square cannot be used and are listed as `excluded_outside_crop`.

In [ ]:
from unetultra.dataset import SessionDataset

train_set = SessionDataset(train_paths, calibration, augment=True)
val_set = SessionDataset(val_paths, calibration)
background = sum(int(train_set[i]["instances"].max()) == 0 for i in range(len(train_set)))
print(f"train {len(train_set)} samples ({train_set.skipped} skipped as not accepted), validation {len(val_set)}")
for path, sample_id, reason in train_set.excluded + val_set.excluded:
    print(f"  {reason}: {Path(path).name} sample {sample_id}")
print(f"background-only share of training samples: {background / max(len(train_set), 1):.0%} (aim for about 10%)")
print("classes:", train_set.classes)

## Train the model

In [ ]:
from unetultra.train import TrainConfig, fit

config = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, depth_weight=DEPTH_WEIGHT,
                     object_depth_weight=OBJECT_DEPTH_WEIGHT)
history, checkpoint = fit(train_set, val_set, config, MODELS, name="model")
print("best checkpoint:", checkpoint)

## Train the image-only ablation

Same data and settings, with the meta input zeroed.

In [ ]:
from dataclasses import replace

ablation_config = replace(config, use_meta=False)
ablation_history, ablation_checkpoint = fit(
    SessionDataset(train_paths, calibration, augment=True, use_meta=False),
    SessionDataset(val_paths, calibration, use_meta=False),
    ablation_config, MODELS, name="model_image_only")

## Learning curves

In [ ]:
plots.learning_curves({"with focus": history, "image only": ablation_history});